In [2]:
import cv2
import numpy as np
import tensorflow as tf
from tqdm import tqdm

In [3]:
IMAGE_PATH = "forest_and_roads.tif"
MODEL_PATH = "landcover_classifier.h5"
PATCH_SIZE = 64
STRIDE = 32
ALPHA = 0.4
BATCH_SIZE = 128

CLASS_COLORS = {
    'forest': (0, 255, 0),
    'road': (0, 0, 255),
}
CLASS_NAMES = ['road', 'forest']

model = tf.keras.models.load_model(MODEL_PATH)

img = cv2.imread(IMAGE_PATH, cv2.IMREAD_COLOR)
if img is None:
    raise RuntimeError(f"Cannot load {IMAGE_PATH}")
h, w = img.shape[:2]
print(f"Image size: {w} x {h}")

num_patches_y = (h - PATCH_SIZE) // STRIDE + 1
num_patches_x = (w - PATCH_SIZE) // STRIDE + 1
total_patches = num_patches_y * num_patches_x
print(f"Total patches: {total_patches}")

result_overlay = img.copy()

def patch_generator():
    for y in range(0, h - PATCH_SIZE + 1, STRIDE):
        for x in range(0, w - PATCH_SIZE + 1, STRIDE):
            patch = img[y:y+PATCH_SIZE, x:x+PATCH_SIZE]
            patch_resized = cv2.resize(patch, (PATCH_SIZE, PATCH_SIZE))
            patch_norm = patch_resized / 255.0
            yield x, y, patch_norm.astype(np.float32)

batch_x = []
batch_y = []
batch_patches = []

pbar = tqdm(total=total_patches, desc="Processing patches")

for x, y, patch_norm in patch_generator():
    batch_x.append(x)
    batch_y.append(y)
    batch_patches.append(patch_norm)
    
    if len(batch_patches) >= BATCH_SIZE:
        batch_input = np.array(batch_patches, dtype=np.float32)
        preds = model.predict(batch_input, verbose=0)
        
        for i in range(len(batch_patches)):
            class_idx = np.argmax(preds[i])
            confidence = np.max(preds[i])
            if confidence >= 0.7:
                class_name = CLASS_NAMES[class_idx]
                color = CLASS_COLORS.get(class_name, (128, 128, 128))
                xx, yy = batch_x[i], batch_y[i]
                
                overlay = result_overlay[yy:yy+PATCH_SIZE, xx:xx+PATCH_SIZE]
                color_overlay = np.full_like(overlay, color, dtype=np.uint8)
                blended = cv2.addWeighted(overlay, 1-ALPHA, color_overlay, ALPHA, 0)
                result_overlay[yy:yy+PATCH_SIZE, xx:xx+PATCH_SIZE] = blended
        
        batch_x = []
        batch_y = []
        batch_patches = []
        pbar.update(BATCH_SIZE)

if batch_patches:
    batch_input = np.array(batch_patches, dtype=np.float32)
    preds = model.predict(batch_input, verbose=0)
    for i in range(len(batch_patches)):
        class_idx = np.argmax(preds[i])
        confidence = np.max(preds[i])
        if confidence >= 0.7:
            class_name = CLASS_NAMES[class_idx]
            color = CLASS_COLORS.get(class_name, (128, 128, 128))
            xx, yy = batch_x[i], batch_y[i]
            
            overlay = result_overlay[yy:yy+PATCH_SIZE, xx:xx+PATCH_SIZE]
            color_overlay = np.full_like(overlay, color, dtype=np.uint8)
            blended = cv2.addWeighted(overlay, 1-ALPHA, color_overlay, ALPHA, 0)
            result_overlay[yy:yy+PATCH_SIZE, xx:xx+PATCH_SIZE] = blended
    pbar.update(len(batch_patches))

pbar.close()

final = cv2.addWeighted(img, 0.6, result_overlay, 0.4, 0)
cv2.imwrite("result_with_overlay.png", final)
print("Result saved as result_with_overlay.png")

Image size: 26542 x 17924
Total patches: 462852


Processing patches: 100%|██████████| 462852/462852 [18:16<00:00, 421.99it/s] 


Result saved as result_with_overlay.png
